# M2: Entrenamiento de Segmentación de Instancias de Vehículos (YOLO11-seg)

Clasifica y segmenta cada vehículo en las 5 categorías del peaje:
- `moto`, `auto`, `minibus`, `camion`, `bus`

Dataset gestionado y ampliado con imágenes propias en **Roboflow** (formato YOLOv8/YOLO11 Segmentation).

In [ ]:
!pip install -q ultralytics==8.3.49 roboflow==1.1.49
import os, json, torch
from roboflow import Roboflow
from ultralytics import YOLO

print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# Guardar ROBOFLOW_API_KEY en Colab Secrets (icono de llave) para NO exponerla en el notebook
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
WORKSPACE = 'tu-workspace-roboflow'
PROJECT_NAME = 'vehiculos-peaje-seg'
VERSION_NUM = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
dataset = project.version(VERSION_NUM).download('yolov11')
print('Dataset descargado en:', dataset.location)

In [ ]:
model = YOLO('yolo11s-seg.pt')
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project='/content/drive/MyDrive/proyecto_peaje/runs_m2',
    name='m2_vehiculos_seg',
    exist_ok=True
)

In [ ]:
best_weights = '/content/drive/MyDrive/proyecto_peaje/runs_m2/m2_vehiculos_seg/weights/best.pt'
best_model = YOLO(best_weights)
metrics = best_model.val(data=f'{dataset.location}/data.yaml')

report = {
    'modelo': 'M2_vehiculos_seg',
    'box_mAP50': round(float(metrics.box.map50), 4),
    'box_mAP50_95': round(float(metrics.box.map), 4),
    'mask_mAP50': round(float(metrics.seg.map50), 4),
    'mask_mAP50_95': round(float(metrics.seg.map), 4),
    'precision': round(float(metrics.seg.mp), 4),
    'recall': round(float(metrics.seg.mr), 4)
}
print(json.dumps(report, indent=2))

# Copiar el peso final al directorio central de modelos para subir a RunPod
os.makedirs('/content/drive/MyDrive/proyecto_peaje/models', exist_ok=True)
!cp /content/drive/MyDrive/proyecto_peaje/runs_m2/m2_vehiculos_seg/weights/best.pt /content/drive/MyDrive/proyecto_peaje/models/m2_vehiculos_seg.pt
with open('/content/drive/MyDrive/proyecto_peaje/metricas_M2.json', 'w') as f:
    json.dump(report, f, indent=2)